# Gigs Senior Data Analyst Challenge

Welcome to the Gigs data analyst take-home challenge! This notebook will help you get started with analyzing our connectivity usage data.

## About the Data

You'll be working with three main datasets:
- **Usage Data**: Detailed usage per subscription period (~100K+ records)
- **Plan Events**: Plan configuration and pricing history
- **Projects**: Project metadata

## Setup Instructions

Run the cells below to set up your environment and load the data into DuckDB.

In [24]:
# Import required libraries
import duckdb
import pandas as pd
from datetime import datetime, timedelta

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


In [25]:
# Load JupySQL extension and configure
%load_ext sql

# Configure JupySQL for better output
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

print("✅ JupySQL configured!")

✅ JupySQL configured!


In [26]:
# Connect to DuckDB
conn = duckdb.connect('gigs-analytics.db')
%sql conn --alias duckdb

print("✅ Connected to DuckDB database: gigs-analytics.db")

✅ Connected to DuckDB database: gigs-analytics.db


In [27]:
%%sql
-- Load data into DuckDB tables
CREATE OR REPLACE TABLE usage_data AS 
SELECT * FROM 'data/usage_by_subscription_period.csv';

CREATE OR REPLACE TABLE plan_events AS 
SELECT * FROM 'data/plan_change_events.csv';

CREATE OR REPLACE TABLE projects AS 
SELECT * FROM 'data/projects.csv';

,Count
0,3


In [28]:
%%sql
-- Verify data loading
select 
  'usage_data' as table_name, 
  count(*) as row_count,
  count(distinct subscription_id) as unique_subscriptions
from usage_data
union all
select 
  'plan_events' as table_name, 
  count(*) as row_count,
  count(distinct plan_id) as unique_plans
from plan_events
union all
select 
  'projects' as table_name, 
  count(*) as row_count,
  count(distinct project_id__hashed) as unique_projects
from projects;

,table_name,row_count,unique_subscriptions
0,usage_data,53565,8457
1,plan_events,209,36
2,projects,3,3


## Your Analysis Starts Here!

Now you have everything set up. Use the cells below to start your analysis.

### Tips:
- Use `%%sql` for multi-line SQL queries
- Use `%sql variable_name <<` to store results in a Python variable
- Combine SQL with Python/Pandas for advanced analysis
- Feel free to use any visualisation library you feel comfortable with

---
---
# Start

#### Additional tools

In [ ]:

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import plotly.express as px


# 1) Loading

In [3]:
pro = pd.read_csv('data/projects.csv')
df = pd.read_csv('data/usage_by_subscription_period.csv')
plan = pd.read_csv('data/plan_change_events.csv')

In [4]:
pro.head()

,project_id__hashed,project_type,organization_name,device_type
0,dace2786aee7632e61757b320a6fe5bff37a2e742fe558...,API,People Mobile,Phones
1,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,Connect,ACME Phone,Phones
2,2aeca1a6c1ecf52b28b7f7646b6fb90563a417ee3e5dc3...,Connect,SmartDevices Inc.,Wearables


In [37]:
pro.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   project_id__hashed  3 non-null      object
 1   project_type        3 non-null      object
 2   organization_name   3 non-null      object
 3   device_type         3 non-null      object
dtypes: object(4)
memory usage: 224.0+ bytes


In [38]:
df.head(2)

,subscription_id,project_id__hashed,plan_id,reporting_date,subscription_period_start,subscription_period_end,subscription_period_number,cumulative_data_usage_megabyte,cumulative_voice_usage_minutes,cumulative_sms_usage,number_of_addons_activated
0,sub_b97107a1c7ef89cf28a7e83ee850,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,pln_0a3b55b9abb575b023c07e617b49,2024-02-07 00:00:00,2024-02-04,2024-02-07,1,0.000000,0.0,0.0,0
1,sub_2353ef9e8bf55916b97ceb30194c,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,pln_0a3b55b9abb575b023c07e617b49,2024-02-08 00:00:00,2024-01-25,2024-02-08,1,56.660992,1.0,1.0,0


In [39]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53565 entries, 0 to 53564
Data columns (total 11 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   subscription_id                 53565 non-null  object 
 1   project_id__hashed              53565 non-null  object 
 2   plan_id                         53565 non-null  object 
 3   reporting_date                  53565 non-null  object 
 4   subscription_period_start       53565 non-null  object 
 5   subscription_period_end         53565 non-null  object 
 6   subscription_period_number      53565 non-null  int64  
 7   cumulative_data_usage_megabyte  53565 non-null  float64
 8   cumulative_voice_usage_minutes  53565 non-null  float64
 9   cumulative_sms_usage            53565 non-null  float64
 10  number_of_addons_activated      53565 non-null  int64  
dtypes: float64(3), int64(2), object(6)
memory usage: 4.5+ MB


In [5]:
plan.head(3)


,plan_id,project_id__hashed,plan_created_at,event_type,event_timestamp,plan_name,network_provider_id,price_currency,plan_price_amount_local,data_allowance_mb,is_unlimited_data,voice_allowance_seconds,is_unlimited_voice,sms_allowance,is_unlimited_sms,validity_value,validity_unit,_valid_from,_valid_to,_is_current_state
0,pln_b2a155624e05108d1a5b6b90f567,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,2023-12-29 18:43:12,plan.updated,2023-12-29 18:44:37.573698,Unlimited Data Plan,p4,USD,70.0,NaN,True,NaN,True,NaN,True,30,day,2023-12-29 18:44:37.573698,2023-12-29 18:44:37.846803,False
1,pln_b2a155624e05108d1a5b6b90f567,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,2023-12-29 18:43:12,plan.archived,2024-01-21 15:48:13.444971,Unlimited Data Plan,p4,USD,70.0,NaN,True,NaN,True,NaN,True,30,day,2024-01-21 15:48:13.444971,NaN,True
2,pln_b2a155624e05108d1a5b6b90f567,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,2023-12-29 18:43:12,plan.updated,2024-01-21 15:48:12.953812,Unlimited Data Plan,p4,USD,70.0,NaN,True,NaN,True,NaN,True,30,day,2024-01-21 15:48:12.953812,2024-01-21 15:48:13.444971,False


In [6]:
plan.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 209 entries, 0 to 208
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   plan_id                  209 non-null    object 
 1   project_id__hashed       209 non-null    object 
 2   plan_created_at          209 non-null    object 
 3   event_type               209 non-null    object 
 4   event_timestamp          209 non-null    object 
 5   plan_name                209 non-null    object 
 6   network_provider_id      209 non-null    object 
 7   price_currency           209 non-null    object 
 8   plan_price_amount_local  209 non-null    float64
 9   data_allowance_mb        120 non-null    float64
 10  is_unlimited_data        209 non-null    bool   
 11  voice_allowance_seconds  41 non-null     float64
 12  is_unlimited_voice       209 non-null    bool   
 13  sms_allowance            35 non-null     float64
 14  is_unlimited_sms         2

In [42]:
plan.duplicated().all(), df.duplicated().all() , pro.duplicated().all()

(np.False_, np.False_, np.False_)

# 2) DATA QUALITY CHECKS



In [ ]:
def show_stats(df):
    for cols in df.columns:
        print(f"{cols} {'-'*(20-len(cols))}:  unique[{df[cols].nunique()}], null: {df[cols].isnull().sum()}, na:{df[cols].isna().sum()} \n samples --> {df[cols].unique()[:3]}", end="\n\n")

print('-----------------------------------------')
print('---------------- project ----------------')
print('-----------------------------------------\n\n')
show_stats(pro)
print('-----------------------------------------')
print('-----------------subscriptions-----------')
print('-----------------------------------------\n\n')
show_stats(df)
print('----------------------------------------------')
print('-----------------plan_change_events-----------')
print('----------------------------------------------\n\n')
show_stats(plan)



-----------------------------------------
---------------- project ----------------
-----------------------------------------


project_id__hashed --:  unique[3], null: 0, na:0 
 samples --> ['dace2786aee7632e61757b320a6fe5bff37a2e742fe5582c245665e3d0e3a84e'
 '82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f3597cb2817a9b5a87bbf3'
 '2aeca1a6c1ecf52b28b7f7646b6fb90563a417ee3e5dc32f9d1a50d7e8b9e13a']

project_type --------:  unique[2], null: 0, na:0 
 samples --> ['API' 'Connect']

organization_name ---:  unique[3], null: 0, na:0 
 samples --> ['People Mobile' 'ACME Phone' 'SmartDevices Inc.']

device_type ---------:  unique[2], null: 0, na:0 
 samples --> ['Phones' 'Wearables']

-----------------------------------------
-----------------subscriptions-----------
-----------------------------------------


subscription_id -----:  unique[8457], null: 0, na:0 
 samples --> ['sub_b97107a1c7ef89cf28a7e83ee850' 'sub_2353ef9e8bf55916b97ceb30194c'
 'sub_5d5605c8cf0b46116a42cedb9561']

project_id__hash

In [ ]:
plan.event_type.unique()

array(['plan.updated', 'plan.archived', 'plan.published', 'plan.created'],
      dtype=object)

## Observations:

In general:

- There are no null values (except and sms allowance , which are not being used)
- There are no duplicates

Project: 

1. Its clean

Subscription: 

1. subscription start and end have different lenght, to check why.
1. cumulative data usage can be 0 
1. same for sms
  
Plan: 
1. voice_allowance_seconds , sms_allowance  could need some cleaning, but here we wont use it
1. plan archived could be use for churn
1. There are plan prices  that are 0 usd



In [30]:
%%sql

# can a subscription have multiple plans? - YES

SELECT 
subscription_id
# count(Distinct plan_id) as total_plans
FROM usage_data
# group by 1
# order by 2 desc 
limit 2; 

,subscription_id
0,sub_b97107a1c7ef89cf28a7e83ee850
1,sub_2353ef9e8bf55916b97ceb30194c


In [31]:
%%sql

# example of a subscription with multiple plans
select 
* 
from usage_data

where subscription_id = 'sub_bf65d729c2df4092017eaea0ee0a';



,subscription_id,project_id__hashed,plan_id,reporting_date,subscription_period_start,subscription_period_end,subscription_period_number,cumulative_data_usage_megabyte,cumulative_voice_usage_minutes,cumulative_sms_usage,number_of_addons_activated
0,sub_bf65d729c2df4092017eaea0ee0a,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,pln_0a3b55b9abb575b023c07e617b49,2024-05-24,2024-04-25,2024-05-25,1,1829.331968,773.0,214.0,0
1,sub_bf65d729c2df4092017eaea0ee0a,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,pln_0a3b55b9abb575b023c07e617b49,2024-06-24,2024-05-25,2024-06-25,2,1488.722944,1761.0,484.0,0
2,sub_bf65d729c2df4092017eaea0ee0a,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,pln_0a3b55b9abb575b023c07e617b49,2024-07-24,2024-06-25,2024-07-25,3,1360.831488,1695.0,1205.0,0
3,sub_bf65d729c2df4092017eaea0ee0a,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,pln_0a3b55b9abb575b023c07e617b49,2024-08-24,2024-07-25,2024-08-25,4,1040.356352,1370.0,340.0,0
4,sub_bf65d729c2df4092017eaea0ee0a,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,pln_0a3b55b9abb575b023c07e617b49,2024-09-24,2024-08-25,2024-09-25,5,1449.269248,1179.0,243.0,0
5,sub_bf65d729c2df4092017eaea0ee0a,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,pln_0a3b55b9abb575b023c07e617b49,2024-10-24,2024-09-25,2024-10-25,6,1337.875456,1093.0,289.0,0
6,sub_bf65d729c2df4092017eaea0ee0a,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,pln_0a3b55b9abb575b023c07e617b49,2024-11-24,2024-10-25,2024-11-25,7,1284.484096,1306.0,443.0,0
7,sub_bf65d729c2df4092017eaea0ee0a,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,pln_af7b4a30086929d37e7f4eb3daed,2024-12-24,2024-11-25,2024-12-25,8,9843.621888,1300.0,553.0,0
8,sub_bf65d729c2df4092017eaea0ee0a,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,pln_1d3f49d61a672755efc1865c4d9a,2025-01-24,2024-12-25,2025-01-25,9,13642.873856,1021.0,229.0,0
9,sub_bf65d729c2df4092017eaea0ee0a,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,pln_1d3f49d61a672755efc1865c4d9a,2025-02-24,2025-01-25,2025-02-25,10,19221.571584,706.0,188.0,0


In [32]:
%%sql

# testing if overlapping periods are possible- YES

select 
subscription_id,
max(subscription_period_number) as max_subscription_period_number,
count(distinct subscription_period_number) as total_subscription_period_number
from usage_data
group by 1
having max_subscription_period_number != total_subscription_period_number;

,subscription_id,max_subscription_period_number,total_subscription_period_number
0,sub_e8658b23ec7d091268689321c938,14,8
1,sub_dc7e2c7861145c0a1aadf1419c9f,4,3
2,sub_1f47cf995a5c20952788526cf706,13,11
3,sub_55eaa81ecba9ed27713987c44a01,8,5
4,sub_713214372cde25f837fb98d8887f,7,6
...,...,...,...
125,sub_5fcce2fe700bf6bb70572b82e722,2,1
126,sub_c1eb9550b3e7bee55a5dfb4315dc,10,7
127,sub_aee18d44bb919e570e2e70523ce8,6,5
128,sub_aab044319bad37030b8172ce984e,9,6


In [33]:
%%sql 

# -- there are overlapping period_numbers  as well as missing periods 

select 
* 
from usage_data

where subscription_id = 'sub_b05686892dcc5e429cde4571596c';





,subscription_id,project_id__hashed,plan_id,reporting_date,subscription_period_start,subscription_period_end,subscription_period_number,cumulative_data_usage_megabyte,cumulative_voice_usage_minutes,cumulative_sms_usage,number_of_addons_activated
0,sub_b05686892dcc5e429cde4571596c,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,pln_0a3b55b9abb575b023c07e617b49,2024-03-29,2024-02-29,2024-03-30,1,342.447104,144.0,172.0,0
1,sub_b05686892dcc5e429cde4571596c,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,pln_0a3b55b9abb575b023c07e617b49,2024-05-30,2024-04-30,2024-05-30,3,0.000000,0.0,0.0,0
2,sub_b05686892dcc5e429cde4571596c,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,pln_0a3b55b9abb575b023c07e617b49,2024-08-29,2024-07-31,2024-08-29,6,0.000000,0.0,0.0,0
3,sub_b05686892dcc5e429cde4571596c,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,pln_0a3b55b9abb575b023c07e617b49,2024-09-30,2024-08-29,2024-09-30,7,0.000000,0.0,0.0,0
4,sub_b05686892dcc5e429cde4571596c,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,pln_0a3b55b9abb575b023c07e617b49,2024-10-21,2024-09-30,2024-10-21,7,0.000000,0.0,0.0,0


In [35]:

%%sql
# can a subscription have multiple projects? - NO

SELECT 
subscription_id,
count(distinct project_id__hashed) as total_projects
FROM usage_data
group by 1

order by 2 desc 
limit 3; 

,subscription_id,total_projects
0,sub_abc7cba1137d81bdd650450d8aaf,1
1,sub_ff09d41e84a767b82a1fdad8d238,1
2,sub_30b93c92422fbcfc84ebea52b5b1,1


# 3) Solving the Questions:

## Q1 : How much data does a subscription typically consume?


Thoughts:
  
This questions is vague thus it depends by the perspective. Unfortunatelly we dont know which team is asking this question (finance, customer service, ...)

TO DO:  
  1. First lets find the data consumption 
  1. Join with the plan using plan_id and project to have all dimensions
  1. Filter to finished/ended subscriptions to avoid ongoing subscriptions
  1. Because there are gaps in the period progression AND repited numbers, use count.  
  1. There could be empty usage months, thus we take out from the calculation
  1. There could be many dimensions to calculate the avg, such as  per project, per plan, device, etc. but since is not explicitly asked we take the globla view.
    
  

In [83]:
%%sql

WITH 

valid_subscriptions AS (

  SELECT 
    subscription_id,
    MAX(reporting_date) as latest_report,
    MAX(subscription_period_end) as last_period_end_date
  FROM
    usage_data
  GROUP BY 
    subscription_id
  HAVING
     MAX(reporting_date) >= MAX(subscription_period_end) 
),

subscription_usage AS (
  
  SELECT 
    a.project_id__hashed,
    a.subscription_id,
    COUNT(a.subscription_period_number) AS count_periods,
    SUM(a.cumulative_data_usage_megabyte) AS sum_data_usage,
    SUM(a.cumulative_data_usage_megabyte) / COUNT(a.subscription_period_number) as avg_usage
  FROM 
    usage_data as a
  INNER JOIN 
    valid_subscriptions as b
    on a.subscription_id = b.subscription_id
  
    
  GROUP BY 1,2
  HAVING sum_data_usage > 0 
  
)
  
  
SELECT 
 sum(sum_data_usage)/sum(count_periods) global_avg,
 median(avg_usage) as median_avg, 
 min(avg_usage) as min_usage,
 max(avg_usage) as max_usage,
 PERCENTILE_CONT(0.25) WITHIN GROUP( ORDER BY sum_data_usage) as p25_usage,
 PERCENTILE_CONT(0.50) WITHIN GROUP( ORDER BY sum_data_usage) as p50_usage,
 PERCENTILE_CONT(0.75) WITHIN GROUP( ORDER BY sum_data_usage) as p75_usage,
 PERCENTILE_CONT(0.95) WITHIN GROUP( ORDER BY sum_data_usage) as p95_usage,
 COUNT(subscription_id) as count_subscriptions,
 COUNT(distinct subscription_id) as count_subscriptions,

FROM
  subscription_usage



,global_avg,median_avg,min_usage,max_usage,p25_usage,p50_usage,p75_usage,p95_usage,count_subscriptions,count_subscriptions_1
0,2127.970248,297.99936,0.001024,211464.075264,257.219654,1295.480832,4842.07104,40173.273907,1695,1695


In [77]:
43/4


10.75

In [75]:
19.338240/6

3.2230399999999997

In [ ]:
# can a subscription have multiple projects?
%%sql
SELECT 

  subscription_id,
  sum(cumulative_data_usage_megabyte) as total_cumulative_data_usage_megabyte,
  avg(cumulative_data_usage_megabyte) as avg_cumulative_data_usage_megabyte

FROM usage_data

group by 1

;



IndentationError: unexpected indent (387840675.py, line 5)

In [43]:
### sandbox

In [48]:
%%sql

# example of a subscription with multiple plans
select 
* 
from projects
limit 5;

,project_id__hashed,project_type,organization_name,device_type
0,dace2786aee7632e61757b320a6fe5bff37a2e742fe558...,API,People Mobile,Phones
1,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,Connect,ACME Phone,Phones
2,2aeca1a6c1ecf52b28b7f7646b6fb90563a417ee3e5dc3...,Connect,SmartDevices Inc.,Wearables


In [59]:
%%sql

# example of a subscription with multiple plans
select 
* 
from plan_events
limit 3;

,plan_id,project_id__hashed,plan_created_at,event_type,event_timestamp,plan_name,network_provider_id,price_currency,plan_price_amount_local,data_allowance_mb,is_unlimited_data,voice_allowance_seconds,is_unlimited_voice,sms_allowance,is_unlimited_sms,validity_value,validity_unit,_valid_from,_valid_to,_is_current_state
0,pln_b2a155624e05108d1a5b6b90f567,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,2023-12-29 18:43:12,plan.updated,2023-12-29 18:44:37.573698,Unlimited Data Plan,p4,USD,70.0,NaN,True,NaN,True,NaN,True,30,day,2023-12-29 18:44:37.573698,2023-12-29 18:44:37.846803,False
1,pln_b2a155624e05108d1a5b6b90f567,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,2023-12-29 18:43:12,plan.archived,2024-01-21 15:48:13.444971,Unlimited Data Plan,p4,USD,70.0,NaN,True,NaN,True,NaN,True,30,day,2024-01-21 15:48:13.444971,NaT,True
2,pln_b2a155624e05108d1a5b6b90f567,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,2023-12-29 18:43:12,plan.updated,2024-01-21 15:48:12.953812,Unlimited Data Plan,p4,USD,70.0,NaN,True,NaN,True,NaN,True,30,day,2024-01-21 15:48:12.953812,2024-01-21 15:48:13.444971,False


In [56]:
%%sql

# example of a subscription with multiple plans
select 
* 
from usage_data
limit 5

,subscription_id,project_id__hashed,plan_id,reporting_date,subscription_period_start,subscription_period_end,subscription_period_number,cumulative_data_usage_megabyte,cumulative_voice_usage_minutes,cumulative_sms_usage,number_of_addons_activated
0,sub_b97107a1c7ef89cf28a7e83ee850,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,pln_0a3b55b9abb575b023c07e617b49,2024-02-07,2024-02-04,2024-02-07,1,0.000000,0.0,0.0,0
1,sub_2353ef9e8bf55916b97ceb30194c,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,pln_0a3b55b9abb575b023c07e617b49,2024-02-08,2024-01-25,2024-02-08,1,56.660992,1.0,1.0,0
2,sub_5d5605c8cf0b46116a42cedb9561,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,pln_1d3f49d61a672755efc1865c4d9a,2024-02-08,2024-01-25,2024-02-08,1,160.659456,0.0,1.0,0
3,sub_8e114d3a638d12ca1526a3fbedd6,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,pln_0a3b55b9abb575b023c07e617b49,2024-02-08,2024-01-28,2024-02-08,1,156.870656,93.0,198.0,0
4,sub_16059484727214a6b903fa16d0fe,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,pln_0a3b55b9abb575b023c07e617b49,2024-02-10,2024-02-09,2024-02-10,1,0.000000,0.0,0.0,0


In [54]:
%%sql

# example of a subscription with multiple plans
select 
* 
from plan_events
limit 5

,plan_id,project_id__hashed,plan_created_at,event_type,event_timestamp,plan_name,network_provider_id,price_currency,plan_price_amount_local,data_allowance_mb,is_unlimited_data,voice_allowance_seconds,is_unlimited_voice,sms_allowance,is_unlimited_sms,validity_value,validity_unit,_valid_from,_valid_to,_is_current_state
0,pln_b2a155624e05108d1a5b6b90f567,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,2023-12-29 18:43:12,plan.updated,2023-12-29 18:44:37.573698,Unlimited Data Plan,p4,USD,70.0,NaN,True,NaN,True,NaN,True,30,day,2023-12-29 18:44:37.573698,2023-12-29 18:44:37.846803,False
1,pln_b2a155624e05108d1a5b6b90f567,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,2023-12-29 18:43:12,plan.archived,2024-01-21 15:48:13.444971,Unlimited Data Plan,p4,USD,70.0,NaN,True,NaN,True,NaN,True,30,day,2024-01-21 15:48:13.444971,NaT,True
2,pln_b2a155624e05108d1a5b6b90f567,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,2023-12-29 18:43:12,plan.updated,2024-01-21 15:48:12.953812,Unlimited Data Plan,p4,USD,70.0,NaN,True,NaN,True,NaN,True,30,day,2024-01-21 15:48:12.953812,2024-01-21 15:48:13.444971,False
3,pln_b2a155624e05108d1a5b6b90f567,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,2023-12-29 18:43:12,plan.published,2023-12-29 18:44:37.846803,Unlimited Data Plan,p4,USD,70.0,NaN,True,NaN,True,NaN,True,30,day,2023-12-29 18:44:37.846803,2024-01-21 15:48:12.953812,False
4,pln_b2a155624e05108d1a5b6b90f567,82728d5d3cf7f35b58bc318399c2c5caf7eeadcc37f359...,2023-12-29 18:43:12,plan.created,2023-12-29 18:43:12.010884,Unlimited Data Plan,p4,USD,70.0,NaN,True,NaN,True,NaN,True,30,day,2023-12-29 18:43:12.000000,2023-12-29 18:44:37.573698,False


In [ ]:


%%sql

# example of a subscription with multiple plans
select 

subscription_id, project_id__hashed, reporting_date,	plan_id,	 cumulative_data_usage_megabyte	


from usage_data

In [ ]:
projects